
# Volume Bucket Schema Reference

## Table of Contents

1. [Overview](#1-overview)
2. [Meta / Structural](#2-meta--structural)
3. [Order & Volume Statistics (Total/Ambiguous)](#3-order--volume-statistics-totalambiguous)
4. [Active Metrics](#4-active-metrics)
    - 4.1 [Active Order Counts](#41-active-order-counts)
    - 4.2 [Active Volumes](#42-active-volumes)
    - 4.3 [Active Volume Log Deltas](#43-active-volume-log-deltas)
    - 4.4 [Active Order Count Log Deltas](#44-active-order-count-log-deltas)
    - 4.5 [Active Imbalance Metrics](#45-active-imbalance-metrics)
    - 4.6 [Active Link Function Transforms](#46-active-link-function-transforms)
    - 4.7 [Active Imbalance Deltas](#47-active-imbalance-deltas)
    - 4.8 [Active Derived Imbalance Metrics](#48-active-derived-imbalance-metrics)
    - 4.9 [Active Price Metrics (VWAP)](#49-active-price-metrics-vwap)
    - 4.10 [Active Weighted Midpoints](#410-active-weighted-midpoints)
    - 4.11 [Active Price Log Deltas](#411-active-price-log-deltas)
    - 4.12 [Active Weighted Midpoint Log Deltas](#412-active-weighted-midpoint-log-deltas)
    - 4.13 [Active Price Range Metrics](#413-active-price-range-metrics)
    - 4.14 [Active Pace Metrics](#414-active-pace-metrics)
    - 4.15 [Active N-Side Inferred Aggregations](#415-active-n-side-inferred-aggregations)
5. [Adjusted Metrics](#5-adjusted-metrics)
    - 5.1 [Adjusted Volumes](#51-adjusted-volumes)
    - 5.2 [Adjusted Imbalance Metrics](#52-adjusted-imbalance-metrics)
    - 5.3 [Adjusted Link Function Transforms](#53-adjusted-link-function-transforms)
    - 5.4 [Adjusted Imbalance Deltas](#54-adjusted-imbalance-deltas)
    - 5.5 [Adjusted Derived Imbalance Metrics](#55-adjusted-derived-imbalance-metrics)
    - 5.6 [Adjusted Price Metrics (VWAP)](#56-adjusted-price-metrics-vwap)
    - 5.7 [Adjusted Weighted Midpoints](#57-adjusted-weighted-midpoints)
    - 5.8 [Adjusted Price Log Deltas](#58-adjusted-price-log-deltas)
    - 5.9 [Adjusted Weighted Midpoint Log Deltas](#59-adjusted-weighted-midpoint-log-deltas)
6. [Passive Metrics](#6-passive-metrics)
    - 6.1 [Passive Midprice](#61-passive-midprice)
    - 6.2 [Passive Midprice Deltas](#62-passive-midprice-deltas)
    - 6.3 [Passive CDF Statistics](#63-passive-cdf-statistics)
    - 6.4 [Passive Volume Classifications](#64-passive-volume-classifications)
    - 6.5 [Passive Imbalance Metrics](#65-passive-imbalance-metrics)
    - 6.6 [Passive Link Function Transforms](#66-passive-link-function-transforms)
    - 6.7 [Passive Imbalance Deltas](#67-passive-imbalance-deltas)
    - 6.8 [Passive Derived Imbalance Metrics](#68-passive-derived-imbalance-metrics)
7. [Divergence Metrics](#7-divergence-metrics)
    - 7.1 [Aggregated Bar-Level Divergences](#71-aggregated-bar-level-divergences)
    - 7.2 [Bucket-Level Divergences](#72-bucket-level-divergences)
    - 7.3 [Divergence Deltas](#73-divergence-deltas)
8. [Temporal / Tempo Structure](#8-temporal--tempo-structure)
    - 8.1 [Time Elapsed Metrics](#81-time-elapsed-metrics)
    - 8.2 [Pace of Contracts Traded](#82-pace-of-contracts-traded)
    - 8.3 [Temporal Log Deltas](#83-temporal-log-deltas)
    - 8.4 [Derived Tempo Metrics](#84-derived-tempo-metrics)
9. [Directional Signals](#9-directional-signals)
    - 9.1 [Active Directional Signals](#91-active-directional-signals)
    - 9.2 [Adjusted Directional Signals](#92-adjusted-directional-signals)
    - 9.3 [Passive Directional Signals](#93-passive-directional-signals)
10. [Volatility and Stability Overlays](#10-volatility-and-stability-overlays)
11. [Inter-Bucket Deltas (Regime Transitions)](#11-inter-bucket-deltas-regime-transitions)
    - 11.1 [Active Inter-Bucket Deltas](#111-active-inter-bucket-deltas)
    - 11.2 [Adjusted Inter-Bucket Deltas](#112-adjusted-inter-bucket-deltas)
    - 11.3 [Second-Order Deltas (Acceleration)](#113-second-order-deltas-acceleration)
12. [Debug / Diagnostic](#12-debug--diagnostic)
13. [Calculation Order](#13-calculation-order)
14. [Edge Case Handling Summary](#14-edge-case-handling-summary)

---

## 1. Overview

### Three-Tier Classification System

The bucket schema adopts a three-tier classification system mirroring the VolumeBar schema:

|Tier|Prefix|Description|Source|
|---|---|---|---|
|**Active**|`active_`|Trades where aggressor side is known from FIX tag 5796|Direct exchange data|
|**Adjusted**|`adjusted_`|Active trades + N-side trades with inferred classification|Active + price-based inference for unknown|
|**Passive**|`passive_`|All trades classified probabilistically from price movement|CDF-based classification|

### Naming Conventions

- Field names do NOT include aggregation method (e.g., `_sum`, `_bucket_level`)
- Documentation clarifies whether a value is:
    - **Bucket-level:** Calculated directly from aggregated raw data
    - **Bar-level statistic:** Mean/std/range of bar-level values
- Use `pace_of_contracts_traded` (not `pace_of_orders`)

### Key Design Principles

1. **Aggregated metrics (bucket-level):** Calculate directly from raw volumes/prices across entire bucket
2. **Distribution metrics (std, range, skew):** Use bar-level values to capture intra-bucket variability
3. **Delta metrics:** Average bar-to-bar changes (represent transitions, not levels)
4. **Link transforms:** Apply to bucket-level ratios; track std of bar-level transforms

---

## 2. Meta / Structural

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`id`|`uint32`|Unique bucket identifier|Assigned sequentially|0|
|`bucket_type`|`U16`|Bucket type|'fixed', 'interval', 'adaptive'|—|
|`num_bars`|`uint32`|Number of bars in bucket|`len(bars)`|0|
|`all_bars_complete`|`bool`|All bars reached target volume|`np.all(bars['bar_complete'])`|False|
|`bar_volume_size`|`uint32`|Target volume per bar|`bars['bar_volume_size'][0]`|—|
|`contract_roll_any`|`bool`|Any contract roll in bucket|`np.any(bars['contract_roll'])`|False|
|`contract_roll_count`|`uint32`|Count of bars with contract roll|`np.sum(bars['contract_roll'])`|0|
|`latest_instrument_id`|`uint32`|Most recent instrument ID|`bars['latest_instrument_id'][-1]`|0|
|`start_ts_ns`|`uint64`|Earliest timestamp in bucket|`np.min(bars['start_ts_ns'])`|0|
|`end_ts_ns`|`uint64`|Latest timestamp in bucket|`np.max(bars['end_ts_ns'])`|0|
|`time_elapsed_ns_total`|`uint64`|Total bucket duration|`end_ts_ns - start_ts_ns`|0|
|`gap_return_any`|`bool`|Any bar flagged gap_return|`np.any(bars['gap_return'])`|False|
|`gap_return_count`|`uint32`|Count of bars with gap_return|`np.sum(bars['gap_return'])`|0|
|`contains_oversized_order_any`|`bool`|Any bar with oversized order|`np.any(bars['contains_oversized_order'])`|False|
|`contains_oversized_order_count`|`uint32`|Count of bars with oversized order|`np.sum(bars['contains_oversized_order'])`|0|

---

## 3. Order & Volume Statistics (Total/Ambiguous)

These metrics include all order types (A + B + N) and remain unprefixed.

### Volume Metrics

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`volume_total_sum`|`uint64`|Total volume in bucket|`np.sum(bars['volume_total'])`|0|
|`volume_total_mean`|`float64`|Mean bar volume|`np.mean(bars['volume_total'])`|NaN if num_bars=0|
|`volume_total_std`|`float64`|Std of bar volumes|`np.std(..., ddof=1)`|0.0 if num_bars≤1|

### Order Count Metrics

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`order_count_sum`|`uint64`|Total orders in bucket|`np.sum(bars['order_count'])`|0|
|`order_count_mean`|`float64`|Mean bar order count|`np.mean(bars['order_count'])`|NaN if num_bars=0|
|`order_count_std`|`float64`|Std of bar order counts|`np.std(..., ddof=1)`|0.0 if num_bars≤1|

### Order Splits

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`order_splits_sum`|`uint64`|Total order splits|`np.sum(bars['order_splits'])`|0|
|`order_splits_mean`|`float64`|Mean bar order splits|`np.mean(bars['order_splits'])`|NaN if num_bars=0|

### Volume Log Deltas

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`delta_volume_total_log_mean`|`float64`|Mean of bar-to-bar log volume deltas|`np.mean(bars['delta_volume_total_log'])`|NaN if insufficient|
|`delta_volume_total_log_std`|`float64`|Std of bar-to-bar log volume deltas|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`delta_volume_total_log_max`|`float64`|Max absolute log volume delta|`np.max(np.abs(...))`|NaN if insufficient|

### Order Count Log Deltas

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`delta_order_count_log_mean`|`float64`|Mean of bar-to-bar log order count deltas|`np.mean(bars['delta_order_count_log'])`|NaN if insufficient|
|`delta_order_count_log_std`|`float64`|Std of bar-to-bar log order count deltas|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`delta_order_count_log_max`|`float64`|Max absolute log order count delta|`np.max(np.abs(...))`|NaN if insufficient|

---

## 4. Active Metrics

Active metrics are derived from trades where the aggressor side is known (FIX tag 5796).

### 4.1 Active Order Counts

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`active_order_count_buy_sum`|`uint64`|Total buy aggressor orders|`np.sum(bars['active_order_count_buy'])`|0|
|`active_order_count_buy_mean`|`float64`|Mean per bar|`np.mean(...)`|NaN if num_bars=0|
|`active_order_count_buy_std`|`float64`|Std across bars|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`active_order_count_sell_sum`|`uint64`|Total sell aggressor orders|`np.sum(bars['active_order_count_sell'])`|0|
|`active_order_count_sell_mean`|`float64`|Mean per bar|`np.mean(...)`|NaN if num_bars=0|
|`active_order_count_sell_std`|`float64`|Std across bars|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`active_order_count_none_sum`|`uint64`|Total N-side orders|`np.sum(bars['active_order_count_none'])`|0|
|`active_order_count_none_mean`|`float64`|Mean per bar|`np.mean(...)`|NaN if num_bars=0|
|`active_order_count_none_std`|`float64`|Std across bars|`np.std(..., ddof=1)`|0.0 if num_bars≤1|

### 4.2 Active Volumes

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`active_volume_buy_sum`|`uint64`|Total buy aggressor volume|`np.sum(bars['active_volume_buy'])`|0|
|`active_volume_buy_mean`|`float64`|Mean per bar|`np.mean(...)`|NaN if num_bars=0|
|`active_volume_buy_std`|`float64`|Std across bars|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`active_volume_sell_sum`|`uint64`|Total sell aggressor volume|`np.sum(bars['active_volume_sell'])`|0|
|`active_volume_sell_mean`|`float64`|Mean per bar|`np.mean(...)`|NaN if num_bars=0|
|`active_volume_sell_std`|`float64`|Std across bars|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`active_volume_none_sum`|`uint64`|Total N-side volume|`np.sum(bars['active_volume_none'])`|0|
|`active_volume_none_mean`|`float64`|Mean per bar|`np.mean(...)`|NaN if num_bars=0|
|`active_volume_none_std`|`float64`|Std across bars|`np.std(..., ddof=1)`|0.0 if num_bars≤1|

### 4.3 Active Volume Log Deltas

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`delta_active_volume_buy_log_mean`|`float64`|Mean of bar-to-bar log deltas|`np.mean(bars['delta_active_volume_buy_log'])`|NaN if insufficient|
|`delta_active_volume_buy_log_std`|`float64`|Std of bar-to-bar log deltas|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`delta_active_volume_buy_log_max`|`float64`|Max absolute log delta|`np.max(np.abs(...))`|NaN if insufficient|
|`delta_active_volume_sell_log_mean`|`float64`|Mean of bar-to-bar log deltas|`np.mean(bars['delta_active_volume_sell_log'])`|NaN if insufficient|
|`delta_active_volume_sell_log_std`|`float64`|Std of bar-to-bar log deltas|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`delta_active_volume_sell_log_max`|`float64`|Max absolute log delta|`np.max(np.abs(...))`|NaN if insufficient|
|`delta_active_volume_none_log_mean`|`float64`|Mean of bar-to-bar log deltas|`np.mean(bars['delta_active_volume_none_log'])`|NaN if insufficient|
|`delta_active_volume_none_log_std`|`float64`|Std of bar-to-bar log deltas|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`delta_active_volume_none_log_max`|`float64`|Max absolute log delta|`np.max(np.abs(...))`|NaN if insufficient|

### 4.4 Active Order Count Log Deltas

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`delta_active_order_count_buy_log_mean`|`float64`|Mean of bar-to-bar log deltas|`np.mean(bars['delta_active_order_count_buy_log'])`|NaN if insufficient|
|`delta_active_order_count_buy_log_std`|`float64`|Std of bar-to-bar log deltas|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`delta_active_order_count_buy_log_max`|`float64`|Max absolute log delta|`np.max(np.abs(...))`|NaN if insufficient|
|`delta_active_order_count_sell_log_mean`|`float64`|Mean of bar-to-bar log deltas|`np.mean(bars['delta_active_order_count_sell_log'])`|NaN if insufficient|
|`delta_active_order_count_sell_log_std`|`float64`|Std of bar-to-bar log deltas|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`delta_active_order_count_sell_log_max`|`float64`|Max absolute log delta|`np.max(np.abs(...))`|NaN if insufficient|
|`delta_active_order_count_none_log_mean`|`float64`|Mean of bar-to-bar log deltas|`np.mean(bars['delta_active_order_count_none_log'])`|NaN if insufficient|
|`delta_active_order_count_none_log_std`|`float64`|Std of bar-to-bar log deltas|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`delta_active_order_count_none_log_max`|`float64`|Max absolute log delta|`np.max(np.abs(...))`|NaN if insufficient|

### 4.5 Active Imbalance Metrics

Bucket-level imbalances calculated directly from aggregated volumes.

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`active_imbalance_signed`|`int64`|Net directional imbalance|`active_volume_buy_sum - active_volume_sell_sum`|0|
|`active_imbalance_abs`|`uint64`|Absolute imbalance|`abs(active_imbalance_signed)`|0|
|`active_imbalance_signed_ratio`|`float64`|Signed imbalance / total active volume|`active_imbalance_signed / (active_volume_buy_sum + active_volume_sell_sum)`|NaN if total=0|
|`active_imbalance_abs_ratio`|`float64`|Abs imbalance / total active volume|`active_imbalance_abs / (active_volume_buy_sum + active_volume_sell_sum)`|NaN if total=0|
|`active_imbalance_buy_ratio`|`float64`|Buy volume / total active volume|`active_volume_buy_sum / (active_volume_buy_sum + active_volume_sell_sum)`|NaN if total=0|
|`active_imbalance_signed_ratio_std`|`float64`|Std of bar-level signed ratios|`np.std(bars['active_imbalance_signed_ratio'], ddof=1)`|0.0 if num_bars≤1|
|`active_imbalance_abs_ratio_std`|`float64`|Std of bar-level abs ratios|`np.std(bars['active_imbalance_abs_ratio'], ddof=1)`|0.0 if num_bars≤1|
|`active_imbalance_buy_ratio_std`|`float64`|Std of bar-level buy ratios|`np.std(bars['active_imbalance_buy_ratio'], ddof=1)`|0.0 if num_bars≤1|

### 4.6 Active Link Function Transforms

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`active_imbalance_signed_ratio_atanh`|`float64`|Arctanh of bucket-level signed ratio|`np.arctanh(np.clip(ratio, -0.9999, 0.9999))`|NaN if ratio NaN|
|`active_imbalance_signed_ratio_atanh_std`|`float64`|Std of bar-level atanh values|`np.std(bars['active_imbalance_signed_ratio_atanh'], ddof=1)`|0.0 if num_bars≤1|
|`active_imbalance_buy_ratio_logit`|`float64`|Logit of bucket-level buy ratio|`np.log(r / (1.0 - r))` where r = clip(ratio, 0.0001, 0.9999)|NaN if ratio NaN|
|`active_imbalance_buy_ratio_logit_std`|`float64`|Std of bar-level logit values|`np.std(bars['active_imbalance_buy_ratio_logit'], ddof=1)`|0.0 if num_bars≤1|
|`active_imbalance_abs_ratio_logit`|`float64`|Logit of bucket-level abs ratio|`np.log(r / (1.0 - r))` where r = clip(ratio, 0.0001, 0.9999)|NaN if ratio NaN|
|`active_imbalance_abs_ratio_logit_std`|`float64`|Std of bar-level logit values|`np.std(bars['active_imbalance_abs_ratio_logit'], ddof=1)`|0.0 if num_bars≤1|

### 4.7 Active Imbalance Deltas

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`delta_active_imbalance_signed_ratio_mean`|`float64`|Mean of bar-to-bar ratio deltas|`np.mean(bars['delta_active_imbalance_signed_ratio'])`|NaN if insufficient|
|`delta_active_imbalance_signed_ratio_std`|`float64`|Std of bar-to-bar ratio deltas|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`delta_active_imbalance_buy_ratio_mean`|`float64`|Mean of bar-to-bar ratio deltas|`np.mean(bars['delta_active_imbalance_buy_ratio'])`|NaN if insufficient|
|`delta_active_imbalance_buy_ratio_std`|`float64`|Std of bar-to-bar ratio deltas|`np.std(..., ddof=1)`|0.0 if num_bars≤1|

### 4.8 Active Derived Imbalance Metrics

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`active_cumulative_signed_imbalance`|`int64`|Alias for active_imbalance_signed|`active_imbalance_signed`|0|
|`active_imbalance_persistence`|`float64`|Fraction of bars where sign matches bucket sign|`count(sign_match) / num_bars`|0.0 if bucket sign=0|
|`active_imbalance_volatility`|`float64`|Std of bar-level signed ratios|Same as `active_imbalance_signed_ratio_std`|0.0 if num_bars≤1|

### 4.9 Active Price Metrics (VWAP)

Bucket-level VWAPs calculated directly from aggregated volumes.

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`active_buy_vwap`|`float64`|Bucket-level buy VWAP|`np.sum(bars['active_buy_vwap'] * bars['active_volume_buy']) / active_volume_buy_sum`|NaN if no buy vol|
|`active_sell_vwap`|`float64`|Bucket-level sell VWAP|`np.sum(bars['active_sell_vwap'] * bars['active_volume_sell']) / active_volume_sell_sum`|NaN if no sell vol|
|`active_none_vwap`|`float64`|Bucket-level none VWAP|`np.sum(bars['active_none_vwap'] * bars['active_volume_none']) / active_volume_none_sum`|NaN if no none vol|
|`active_spread_vwap`|`float64`|Buy VWAP - Sell VWAP|`active_buy_vwap - active_sell_vwap`|NaN if either NaN|
|`active_midpoint_vwap`|`float64`|(Buy VWAP + Sell VWAP) * 0.5|`(active_buy_vwap + active_sell_vwap) * 0.5`|NaN if either NaN|
|`active_midpoint_vwap_std`|`float64`|Std of bar-level midpoints|`np.std(bars['active_midpoint_vwap'], ddof=1)`|0.0 if num_bars≤1|
|`active_midpoint_vwap_range`|`float64`|Max - min of bar-level midpoints|`np.max(...) - np.min(...)`|0.0 if num_bars≤1|
|`active_spread_vwap_std`|`float64`|Std of bar-level spreads|`np.std(bars['active_spread_vwap'], ddof=1)`|0.0 if num_bars≤1|
|`active_spread_vwap_range`|`float64`|Max - min of bar-level spreads|`np.max(...) - np.min(...)`|0.0 if num_bars≤1|

### 4.10 Active Weighted Midpoints

Bucket-level weighted midpoints calculated from aggregated data.

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`active_mid_imbalance_weighted`|`float64`|Midpoint + (spread * 0.5 * imbalance_ratio)|`active_midpoint_vwap + (active_spread_vwap * 0.5 * np.clip(active_imbalance_signed_ratio, -1, 1))`|NaN if deps NaN|
|`active_mid_flow_weighted`|`float64`|Same-side volume weighting|`(active_sell_vwap * active_volume_sell_sum + active_buy_vwap * active_volume_buy_sum) / (active_volume_buy_sum + active_volume_sell_sum)`|NaN if no vol|
|`active_mid_aggressor_weighted`|`float64`|Opposite-side volume weighting|`(active_sell_vwap * active_volume_buy_sum + active_buy_vwap * active_volume_sell_sum) / (active_volume_buy_sum + active_volume_sell_sum)`|NaN if no vol|
|`active_mid_imbalance_weighted_std`|`float64`|Std of bar-level values|`np.std(bars['active_mid_imbalance_weighted'], ddof=1)`|0.0 if num_bars≤1|
|`active_mid_flow_weighted_std`|`float64`|Std of bar-level values|`np.std(bars['active_mid_flow_weighted'], ddof=1)`|0.0 if num_bars≤1|
|`active_mid_aggressor_weighted_std`|`float64`|Std of bar-level values|`np.std(bars['active_mid_aggressor_weighted'], ddof=1)`|0.0 if num_bars≤1|

### 4.11 Active Price Log Deltas

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`delta_active_midpoint_vwap_log_sum`|`float64`|Sum of bar-to-bar log deltas (bucket return)|`np.sum(bars['delta_active_midpoint_vwap_log'])`|NaN if insufficient|
|`delta_active_midpoint_vwap_log_mean`|`float64`|Mean of bar-to-bar log deltas|`np.mean(bars['delta_active_midpoint_vwap_log'])`|NaN if insufficient|
|`delta_active_midpoint_vwap_log_std`|`float64`|Std of bar-to-bar log deltas|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`delta_active_midpoint_vwap_log_skew`|`float64`|Skewness of bar-to-bar log deltas|`scipy.stats.skew(...)`|0.0 if num_bars≤2|
|`delta_active_spread_vwap_log_mean`|`float64`|Mean of bar-to-bar log deltas|`np.mean(bars['delta_active_spread_vwap_log'])`|NaN if insufficient|
|`delta_active_spread_vwap_log_std`|`float64`|Std of bar-to-bar log deltas|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`delta_active_buy_vwap_log_mean`|`float64`|Mean of bar-to-bar log deltas|`np.mean(bars['delta_active_buy_vwap_log'])`|NaN if insufficient|
|`delta_active_buy_vwap_log_std`|`float64`|Std of bar-to-bar log deltas|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`delta_active_sell_vwap_log_mean`|`float64`|Mean of bar-to-bar log deltas|`np.mean(bars['delta_active_sell_vwap_log'])`|NaN if insufficient|
|`delta_active_sell_vwap_log_std`|`float64`|Std of bar-to-bar log deltas|`np.std(..., ddof=1)`|0.0 if num_bars≤1|

### 4.12 Active Weighted Midpoint Log Deltas

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`delta_active_mid_imbalance_weighted_log_sum`|`float64`|Sum of log deltas|`np.sum(bars['delta_active_mid_imbalance_weighted_log'])`|NaN if insufficient|
|`delta_active_mid_imbalance_weighted_log_mean`|`float64`|Mean of log deltas|`np.mean(...)`|NaN if insufficient|
|`delta_active_mid_imbalance_weighted_log_std`|`float64`|Std of log deltas|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`delta_active_mid_imbalance_weighted_log_skew`|`float64`|Skewness of log deltas|`scipy.stats.skew(...)`|0.0 if num_bars≤2|
|`delta_active_mid_flow_weighted_log_sum`|`float64`|Sum of log deltas|`np.sum(bars['delta_active_mid_flow_weighted_log'])`|NaN if insufficient|
|`delta_active_mid_flow_weighted_log_mean`|`float64`|Mean of log deltas|`np.mean(...)`|NaN if insufficient|
|`delta_active_mid_flow_weighted_log_std`|`float64`|Std of log deltas|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`delta_active_mid_flow_weighted_log_skew`|`float64`|Skewness of log deltas|`scipy.stats.skew(...)`|0.0 if num_bars≤2|
|`delta_active_mid_aggressor_weighted_log_sum`|`float64`|Sum of log deltas|`np.sum(bars['delta_active_mid_aggressor_weighted_log'])`|NaN if insufficient|
|`delta_active_mid_aggressor_weighted_log_mean`|`float64`|Mean of log deltas|`np.mean(...)`|NaN if insufficient|
|`delta_active_mid_aggressor_weighted_log_std`|`float64`|Std of log deltas|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`delta_active_mid_aggressor_weighted_log_skew`|`float64`|Skewness of log deltas|`scipy.stats.skew(...)`|0.0 if num_bars≤2|

### 4.13 Active Price Range Metrics

#### Bucket-Level Extremes (True Min/Max Across All Bars)

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`active_buy_price_min`|`float64`|Min buy price across bucket|`np.nanmin(bars['active_buy_price_min'])`|NaN if no buy trades|
|`active_buy_price_max`|`float64`|Max buy price across bucket|`np.nanmax(bars['active_buy_price_max'])`|NaN if no buy trades|
|`active_buy_price_range`|`float64`|Max - min buy price|`active_buy_price_max - active_buy_price_min`|NaN if no buy trades|
|`active_sell_price_min`|`float64`|Min sell price across bucket|`np.nanmin(bars['active_sell_price_min'])`|NaN if no sell trades|
|`active_sell_price_max`|`float64`|Max sell price across bucket|`np.nanmax(bars['active_sell_price_max'])`|NaN if no sell trades|
|`active_sell_price_range`|`float64`|Max - min sell price|`active_sell_price_max - active_sell_price_min`|NaN if no sell trades|
|`active_none_price_min`|`float64`|Min none price across bucket|`np.nanmin(bars['active_none_price_min'])`|NaN if no none trades|
|`active_none_price_max`|`float64`|Max none price across bucket|`np.nanmax(bars['active_none_price_max'])`|NaN if no none trades|
|`active_none_price_range`|`float64`|Max - min none price|`active_none_price_max - active_none_price_min`|NaN if no none trades|

#### Bar-Level Range Statistics

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`active_buy_price_range_mean`|`float64`|Mean of bar-level buy ranges|`np.nanmean(bars['active_buy_price_range'])`|NaN if insufficient|
|`active_buy_price_range_std`|`float64`|Std of bar-level buy ranges|`np.nanstd(..., ddof=1)`|0.0 if num_bars≤1|
|`active_sell_price_range_mean`|`float64`|Mean of bar-level sell ranges|`np.nanmean(bars['active_sell_price_range'])`|NaN if insufficient|
|`active_sell_price_range_std`|`float64`|Std of bar-level sell ranges|`np.nanstd(..., ddof=1)`|0.0 if num_bars≤1|
|`active_none_price_range_mean`|`float64`|Mean of bar-level none ranges|`np.nanmean(bars['active_none_price_range'])`|NaN if insufficient|
|`active_none_price_range_std`|`float64`|Std of bar-level none ranges|`np.nanstd(..., ddof=1)`|0.0 if num_bars≤1|

### 4.14 Active Pace Metrics

#### Bucket-Level Pace

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`active_buy_pace`|`float64`|Buy volume / total time|`active_volume_buy_sum / time_elapsed_ns_total`|NaN if time=0|
|`active_sell_pace`|`float64`|Sell volume / total time|`active_volume_sell_sum / time_elapsed_ns_total`|NaN if time=0|

#### Bar-Level Statistics

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`active_buy_pace_mean`|`float64`|Mean of bar-level buy pace|`np.nanmean(bars['active_buy_pace'])`|NaN if insufficient|
|`active_buy_pace_std`|`float64`|Std of bar-level buy pace|`np.nanstd(..., ddof=1)`|0.0 if num_bars≤1|
|`active_sell_pace_mean`|`float64`|Mean of bar-level sell pace|`np.nanmean(bars['active_sell_pace'])`|NaN if insufficient|
|`active_sell_pace_std`|`float64`|Std of bar-level sell pace|`np.nanstd(..., ddof=1)`|0.0 if num_bars≤1|

#### Log-Transformed Pace Deltas

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`delta_active_buy_pace_log_mean`|`float64`|Mean of bar-to-bar log deltas|`np.nanmean(bars['delta_active_buy_pace_log'])`|NaN if insufficient|
|`delta_active_buy_pace_log_std`|`float64`|Std of bar-to-bar log deltas|`np.nanstd(..., ddof=1)`|0.0 if num_bars≤1|
|`delta_active_sell_pace_log_mean`|`float64`|Mean of bar-to-bar log deltas|`np.nanmean(bars['delta_active_sell_pace_log'])`|NaN if insufficient|
|`delta_active_sell_pace_log_std`|`float64`|Std of bar-to-bar log deltas|`np.nanstd(..., ddof=1)`|0.0 if num_bars≤1|

### 4.15 Active N-Side Inferred Aggregations

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`active_none_inferred_buy_volume_sum`|`float64`|Total inferred buy volume|`np.sum(bars['active_none_inferred_buy_volume'])`|0.0|
|`active_none_inferred_sell_volume_sum`|`float64`|Total inferred sell volume|`np.sum(bars['active_none_inferred_sell_volume'])`|0.0|
|`active_none_inferred_buy_vwap`|`float64`|Bucket-level inferred buy VWAP|`np.sum(bars['active_none_inferred_buy_vwap'] * bars['active_none_inferred_buy_volume']) / active_none_inferred_buy_volume_sum`|NaN if no vol|
|`active_none_inferred_sell_vwap`|`float64`|Bucket-level inferred sell VWAP|`np.sum(bars['active_none_inferred_sell_vwap'] * bars['active_none_inferred_sell_volume']) / active_none_inferred_sell_volume_sum`|NaN if no vol|

---

## 5. Adjusted Metrics

Adjusted metrics combine active trades with N-side inferred classifications.

### 5.1 Adjusted Volumes

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`adjusted_volume_buy_sum`|`float64`|Active buy + inferred buy|`active_volume_buy_sum + active_none_inferred_buy_volume_sum`|0.0|
|`adjusted_volume_sell_sum`|`float64`|Active sell + inferred sell|`active_volume_sell_sum + active_none_inferred_sell_volume_sum`|0.0|
|`adjusted_volume_buy_mean`|`float64`|Mean per bar|`np.mean(bars['adjusted_volume_buy'])`|NaN if num_bars=0|
|`adjusted_volume_buy_std`|`float64`|Std across bars|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`adjusted_volume_sell_mean`|`float64`|Mean per bar|`np.mean(bars['adjusted_volume_sell'])`|NaN if num_bars=0|
|`adjusted_volume_sell_std`|`float64`|Std across bars|`np.std(..., ddof=1)`|0.0 if num_bars≤1|

### 5.2 Adjusted Imbalance Metrics

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`adjusted_imbalance_signed`|`float64`|Net directional imbalance|`adjusted_volume_buy_sum - adjusted_volume_sell_sum`|0.0|
|`adjusted_imbalance_abs`|`float64`|Absolute imbalance|`abs(adjusted_imbalance_signed)`|0.0|
|`adjusted_imbalance_signed_ratio`|`float64`|Signed imbalance / total volume|`adjusted_imbalance_signed / volume_total_sum`|NaN if total=0|
|`adjusted_imbalance_abs_ratio`|`float64`|Abs imbalance / total volume|`adjusted_imbalance_abs / volume_total_sum`|NaN if total=0|
|`adjusted_imbalance_buy_ratio`|`float64`|Adjusted buy / total volume|`adjusted_volume_buy_sum / volume_total_sum`|NaN if total=0|
|`adjusted_imbalance_signed_ratio_std`|`float64`|Std of bar-level values|`np.std(bars['adjusted_imbalance_signed_ratio'], ddof=1)`|0.0 if num_bars≤1|
|`adjusted_imbalance_abs_ratio_std`|`float64`|Std of bar-level values|`np.std(bars['adjusted_imbalance_abs_ratio'], ddof=1)`|0.0 if num_bars≤1|
|`adjusted_imbalance_buy_ratio_std`|`float64`|Std of bar-level values|`np.std(bars['adjusted_imbalance_buy_ratio'], ddof=1)`|0.0 if num_bars≤1|

### 5.3 Adjusted Link Function Transforms

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`adjusted_imbalance_signed_ratio_atanh`|`float64`|Arctanh of bucket-level ratio|`np.arctanh(np.clip(ratio, -0.9999, 0.9999))`|NaN if ratio NaN|
|`adjusted_imbalance_signed_ratio_atanh_std`|`float64`|Std of bar-level values|`np.std(bars['adjusted_imbalance_signed_ratio_atanh'], ddof=1)`|0.0 if num_bars≤1|
|`adjusted_imbalance_buy_ratio_logit`|`float64`|Logit of bucket-level ratio|`np.log(r / (1.0 - r))`|NaN if ratio NaN|
|`adjusted_imbalance_buy_ratio_logit_std`|`float64`|Std of bar-level values|`np.std(bars['adjusted_imbalance_buy_ratio_logit'], ddof=1)`|0.0 if num_bars≤1|
|`adjusted_imbalance_abs_ratio_logit`|`float64`|Logit of bucket-level ratio|`np.log(r / (1.0 - r))`|NaN if ratio NaN|
|`adjusted_imbalance_abs_ratio_logit_std`|`float64`|Std of bar-level values|`np.std(bars['adjusted_imbalance_abs_ratio_logit'], ddof=1)`|0.0 if num_bars≤1|

### 5.4 Adjusted Imbalance Deltas

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`delta_adjusted_imbalance_signed_ratio_mean`|`float64`|Mean of bar-to-bar deltas|`np.mean(bars['delta_adjusted_imbalance_signed_ratio'])`|NaN if insufficient|
|`delta_adjusted_imbalance_signed_ratio_std`|`float64`|Std of bar-to-bar deltas|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`delta_adjusted_imbalance_buy_ratio_mean`|`float64`|Mean of bar-to-bar deltas|`np.mean(bars['delta_adjusted_imbalance_buy_ratio'])`|NaN if insufficient|
|`delta_adjusted_imbalance_buy_ratio_std`|`float64`|Std of bar-to-bar deltas|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`delta_adjusted_imbalance_signed_ratio_skew`|`float64`|Skewness of bar-to-bar deltas|`scipy.stats.skew(...)`|0.0 if num_bars≤2|

### 5.5 Adjusted Derived Imbalance Metrics

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`adjusted_cumulative_signed_imbalance`|`float64`|Alias for adjusted_imbalance_signed|`adjusted_imbalance_signed`|0.0|
|`adjusted_imbalance_persistence`|`float64`|Fraction of bars matching bucket sign|`count(sign_match) / num_bars`|0.0 if sign=0|
|`adjusted_imbalance_volatility`|`float64`|Std of bar-level signed ratios|Same as `adjusted_imbalance_signed_ratio_std`|0.0 if num_bars≤1|

### 5.6 Adjusted Price Metrics (VWAP)

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`adjusted_buy_vwap`|`float64`|Bucket-level adjusted buy VWAP|`(active_buy_vwap * active_volume_buy_sum + active_none_inferred_buy_vwap * active_none_inferred_buy_volume_sum) / adjusted_volume_buy_sum`|NaN if no vol|
|`adjusted_sell_vwap`|`float64`|Bucket-level adjusted sell VWAP|`(active_sell_vwap * active_volume_sell_sum + active_none_inferred_sell_vwap * active_none_inferred_sell_volume_sum) / adjusted_volume_sell_sum`|NaN if no vol|
|`adjusted_spread_vwap`|`float64`|Buy - Sell|`adjusted_buy_vwap - adjusted_sell_vwap`|NaN if either NaN|
|`adjusted_midpoint_vwap`|`float64`|(Buy + Sell) * 0.5|`(adjusted_buy_vwap + adjusted_sell_vwap) * 0.5`|NaN if either NaN|
|`adjusted_midpoint_vwap_std`|`float64`|Std of bar-level values|`np.std(bars['adjusted_midpoint_vwap'], ddof=1)`|0.0 if num_bars≤1|
|`adjusted_midpoint_vwap_range`|`float64`|Max - min bar-level|`np.max(...) - np.min(...)`|0.0 if num_bars≤1|
|`adjusted_spread_vwap_std`|`float64`|Std of bar-level values|`np.std(bars['adjusted_spread_vwap'], ddof=1)`|0.0 if num_bars≤1|
|`adjusted_spread_vwap_range`|`float64`|Max - min bar-level|`np.max(...) - np.min(...)`|0.0 if num_bars≤1|

### 5.7 Adjusted Weighted Midpoints

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`adjusted_mid_imbalance_weighted`|`float64`|Bucket-level imbalance-weighted mid|`adjusted_midpoint_vwap + (adjusted_spread_vwap * 0.5 * np.clip(adjusted_imbalance_signed_ratio, -1, 1))`|NaN if deps NaN|
|`adjusted_mid_flow_weighted`|`float64`|Bucket-level flow-weighted mid|`(adjusted_sell_vwap * adjusted_volume_sell_sum + adjusted_buy_vwap * adjusted_volume_buy_sum) / (adjusted_volume_buy_sum + adjusted_volume_sell_sum)`|NaN if no vol|
|`adjusted_mid_aggressor_weighted`|`float64`|Bucket-level aggressor-weighted mid|`(adjusted_sell_vwap * adjusted_volume_buy_sum + adjusted_buy_vwap * adjusted_volume_sell_sum) / (adjusted_volume_buy_sum + adjusted_volume_sell_sum)`|NaN if no vol|
|`adjusted_mid_imbalance_weighted_std`|`float64`|Std of bar-level values|`np.std(bars['adjusted_mid_imbalance_weighted'], ddof=1)`|0.0 if num_bars≤1|
|`adjusted_mid_flow_weighted_std`|`float64`|Std of bar-level values|`np.std(bars['adjusted_mid_flow_weighted'], ddof=1)`|0.0 if num_bars≤1|
|`adjusted_mid_aggressor_weighted_std`|`float64`|Std of bar-level values|`np.std(bars['adjusted_mid_aggressor_weighted'], ddof=1)`|0.0 if num_bars≤1|

### 5.8 Adjusted Price Log Deltas

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`delta_adjusted_midpoint_vwap_log_sum`|`float64`|Sum of log deltas|`np.sum(bars['delta_adjusted_midpoint_vwap_log'])`|NaN if insufficient|
|`delta_adjusted_midpoint_vwap_log_mean`|`float64`|Mean of log deltas|`np.mean(...)`|NaN if insufficient|
|`delta_adjusted_midpoint_vwap_log_std`|`float64`|Std of log deltas|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`delta_adjusted_midpoint_vwap_log_skew`|`float64`|Skewness|`scipy.stats.skew(...)`|0.0 if num_bars≤2|
|`delta_adjusted_spread_vwap_log_mean`|`float64`|Mean of log deltas|`np.mean(...)`|NaN if insufficient|
|`delta_adjusted_spread_vwap_log_std`|`float64`|Std of log deltas|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`delta_adjusted_buy_vwap_log_mean`|`float64`|Mean of log deltas|`np.mean(...)`|NaN if insufficient|
|`delta_adjusted_buy_vwap_log_std`|`float64`|Std of log deltas|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`delta_adjusted_sell_vwap_log_mean`|`float64`|Mean of log deltas|`np.mean(...)`|NaN if insufficient|
|`delta_adjusted_sell_vwap_log_std`|`float64`|Std of log deltas|`np.std(..., ddof=1)`|0.0 if num_bars≤1|

### 5.9 Adjusted Weighted Midpoint Log Deltas

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`delta_adjusted_mid_imbalance_weighted_log_sum`|`float64`|Sum|`np.sum(bars['delta_adjusted_mid_imbalance_weighted_log'])`|NaN if insufficient|
|`delta_adjusted_mid_imbalance_weighted_log_mean`|`float64`|Mean|`np.mean(...)`|NaN if insufficient|
|`delta_adjusted_mid_imbalance_weighted_log_std`|`float64`|Std|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`delta_adjusted_mid_imbalance_weighted_log_skew`|`float64`|Skewness|`scipy.stats.skew(...)`|0.0 if num_bars≤2|
|`delta_adjusted_mid_flow_weighted_log_sum`|`float64`|Sum|`np.sum(bars['delta_adjusted_mid_flow_weighted_log'])`|NaN if insufficient|
|`delta_adjusted_mid_flow_weighted_log_mean`|`float64`|Mean|`np.mean(...)`|NaN if insufficient|
|`delta_adjusted_mid_flow_weighted_log_std`|`float64`|Std|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`delta_adjusted_mid_flow_weighted_log_skew`|`float64`|Skewness|`scipy.stats.skew(...)`|0.0 if num_bars≤2|
|`delta_adjusted_mid_aggressor_weighted_log_sum`|`float64`|Sum|`np.sum(bars['delta_adjusted_mid_aggressor_weighted_log'])`|NaN if insufficient|
|`delta_adjusted_mid_aggressor_weighted_log_mean`|`float64`|Mean|`np.mean(...)`|NaN if insufficient|
|`delta_adjusted_mid_aggressor_weighted_log_std`|`float64`|Std|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`delta_adjusted_mid_aggressor_weighted_log_skew`|`float64`|Skewness|`scipy.stats.skew(...)`|0.0 if num_bars≤2|

---

## 6. Passive Metrics

Passive metrics treat all trades equally, using CDF-based probabilistic classification from price movement.

### 6.1 Passive Midprice

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`passive_midprice`|`float64`|Bucket-level VWAP of all trades|`np.sum(bars['passive_midprice'] * bars['volume_total']) / volume_total_sum`|NaN if no vol|
|`passive_midprice_std`|`float64`|Std of bar-level values|`np.std(bars['passive_midprice'], ddof=1)`|0.0 if num_bars≤1|
|`passive_midprice_range`|`float64`|Max - min bar-level|`np.max(...) - np.min(...)`|0.0 if num_bars≤1|

### 6.2 Passive Midprice Deltas

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`delta_passive_midprice_log_sum`|`float64`|Sum of bar-to-bar log deltas|`np.sum(bars['passive_midprice_delta_log'])`|NaN if insufficient|
|`delta_passive_midprice_log_mean`|`float64`|Mean of log deltas|`np.mean(...)`|NaN if insufficient|
|`delta_passive_midprice_log_std`|`float64`|Std of log deltas|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`delta_passive_midprice_log_skew`|`float64`|Skewness|`scipy.stats.skew(...)`|0.0 if num_bars≤2|

### 6.3 Passive CDF Statistics

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`passive_midprice_delta_cdf_mean`|`float64`|Mean of bar-level CDF values|`np.mean(bars['passive_midprice_delta_cdf'])`|NaN if insufficient|
|`passive_midprice_delta_cdf_std`|`float64`|Std of bar-level CDF values|`np.std(..., ddof=1)`|0.0 if num_bars≤1|

### 6.4 Passive Volume Classifications

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`passive_buy_volume_sum`|`float64`|Total inferred buy volume|`np.sum(bars['passive_buy_volume'])`|0.0|
|`passive_sell_volume_sum`|`float64`|Total inferred sell volume|`np.sum(bars['passive_sell_volume'])`|0.0|
|`passive_buy_volume_mean`|`float64`|Mean per bar|`np.mean(...)`|NaN if num_bars=0|
|`passive_buy_volume_std`|`float64`|Std across bars|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`passive_sell_volume_mean`|`float64`|Mean per bar|`np.mean(...)`|NaN if num_bars=0|
|`passive_sell_volume_std`|`float64`|Std across bars|`np.std(..., ddof=1)`|0.0 if num_bars≤1|

### 6.5 Passive Imbalance Metrics

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`passive_imbalance_signed`|`float64`|Buy - sell|`passive_buy_volume_sum - passive_sell_volume_sum`|0.0|
|`passive_imbalance_abs`|`float64`|Absolute|`abs(passive_imbalance_signed)`|0.0|
|`passive_imbalance_signed_ratio`|`float64`|Signed / total|`passive_imbalance_signed / volume_total_sum`|NaN if total=0|
|`passive_imbalance_abs_ratio`|`float64`|Abs / total|`passive_imbalance_abs / volume_total_sum`|NaN if total=0|
|`passive_imbalance_buy_ratio`|`float64`|Buy / total|`passive_buy_volume_sum / volume_total_sum`|NaN if total=0|
|`passive_imbalance_signed_ratio_std`|`float64`|Std of bar-level|`np.std(bars['passive_imbalance_signed_ratio'], ddof=1)`|0.0 if num_bars≤1|
|`passive_imbalance_abs_ratio_std`|`float64`|Std of bar-level|`np.std(bars['passive_imbalance_abs_ratio'], ddof=1)`|0.0 if num_bars≤1|
|`passive_imbalance_buy_ratio_std`|`float64`|Std of bar-level|`np.std(bars['passive_imbalance_buy_ratio'], ddof=1)`|0.0 if num_bars≤1|

### 6.6 Passive Link Function Transforms

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`passive_imbalance_signed_ratio_atanh`|`float64`|Arctanh of bucket-level|`np.arctanh(np.clip(ratio, -0.9999, 0.9999))`|NaN if ratio NaN|
|`passive_imbalance_signed_ratio_atanh_std`|`float64`|Std of bar-level|`np.std(bars['passive_imbalance_signed_ratio_atanh'], ddof=1)`|0.0 if num_bars≤1|
|`passive_imbalance_buy_ratio_logit`|`float64`|Logit of bucket-level|`np.log(r / (1.0 - r))`|NaN if ratio NaN|
|`passive_imbalance_buy_ratio_logit_std`|`float64`|Std of bar-level|`np.std(bars['passive_imbalance_buy_ratio_logit'], ddof=1)`|0.0 if num_bars≤1|
|`passive_imbalance_abs_ratio_logit`|`float64`|Logit of bucket-level|`np.log(r / (1.0 - r))`|NaN if ratio NaN|
|`passive_imbalance_abs_ratio_logit_std`|`float64`|Std of bar-level|`np.std(bars['passive_imbalance_abs_ratio_logit'], ddof=1)`|0.0 if num_bars≤1|

### 6.7 Passive Imbalance Deltas

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`delta_passive_imbalance_signed_ratio_mean`|`float64`|Mean of deltas|`np.mean(bars['delta_passive_imbalance_signed_ratio'])`|NaN if insufficient|
|`delta_passive_imbalance_signed_ratio_std`|`float64`|Std of deltas|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`delta_passive_imbalance_signed_ratio_skew`|`float64`|Skewness|`scipy.stats.skew(...)`|0.0 if num_bars≤2|
|`delta_passive_imbalance_buy_ratio_mean`|`float64`|Mean of deltas|`np.mean(bars['delta_passive_imbalance_buy_ratio'])`|NaN if insufficient|
|`delta_passive_imbalance_buy_ratio_std`|`float64`|Std of deltas|`np.std(..., ddof=1)`|0.0 if num_bars≤1|

### 6.8 Passive Derived Imbalance Metrics

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`passive_cumulative_signed_imbalance`|`float64`|Alias for passive_imbalance_signed|`passive_imbalance_signed`|0.0|
|`passive_imbalance_persistence`|`float64`|Fraction bars matching sign|`count(sign_match) / num_bars`|0.0 if sign=0|
|`passive_imbalance_volatility`|`float64`|Std of bar-level signed ratios|Same as `passive_imbalance_signed_ratio_std`|0.0 if num_bars≤1|

---

## 7. Divergence Metrics

Divergence metrics compare active (aggressor-known) classification to passive (price-inferred) classification.

### 7.1 Aggregated Bar-Level Divergences

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`divergence_buy_volume_mean`|`float64`|Mean of bar-level divergences|`np.mean(bars['divergence_buy_volume'])`|NaN if insufficient|
|`divergence_buy_volume_std`|`float64`|Std of bar-level divergences|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`divergence_sell_volume_mean`|`float64`|Mean of bar-level divergences|`np.mean(bars['divergence_sell_volume'])`|NaN if insufficient|
|`divergence_sell_volume_std`|`float64`|Std of bar-level divergences|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`divergence_imbalance_signed_mean`|`float64`|Mean of bar-level divergences|`np.mean(bars['divergence_imbalance_signed'])`|NaN if insufficient|
|`divergence_imbalance_signed_std`|`float64`|Std of bar-level divergences|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`divergence_imbalance_signed_ratio_mean`|`float64`|Mean of bar-level divergences|`np.mean(bars['divergence_imbalance_signed_ratio'])`|NaN if insufficient|
|`divergence_imbalance_signed_ratio_std`|`float64`|Std of bar-level divergences|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`divergence_buy_ratio_mean`|`float64`|Mean of bar-level divergences|`np.mean(bars['divergence_buy_ratio'])`|NaN if insufficient|
|`divergence_buy_ratio_std`|`float64`|Std of bar-level divergences|`np.std(..., ddof=1)`|0.0 if num_bars≤1|

### 7.2 Bucket-Level Divergences

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`divergence_buy_volume_bucket`|`float64`|Active buy sum - passive buy sum|`active_volume_buy_sum - passive_buy_volume_sum`|NaN if either NaN|
|`divergence_sell_volume_bucket`|`float64`|Active sell sum - passive sell sum|`active_volume_sell_sum - passive_sell_volume_sum`|NaN if either NaN|
|`divergence_imbalance_signed_bucket`|`float64`|Active signed - passive signed|`active_imbalance_signed - passive_imbalance_signed`|NaN if either NaN|
|`divergence_imbalance_signed_ratio_bucket`|`float64`|Active ratio - passive ratio|`active_imbalance_signed_ratio - passive_imbalance_signed_ratio`|NaN if either NaN|
|`divergence_buy_ratio_bucket`|`float64`|Active buy ratio - passive buy ratio|`active_imbalance_buy_ratio - passive_imbalance_buy_ratio`|NaN if either NaN|

### 7.3 Divergence Deltas

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`delta_divergence_buy_volume_mean`|`float64`|Mean of bar-to-bar deltas|`np.mean(bars['delta_divergence_buy_volume'])`|NaN if insufficient|
|`delta_divergence_sell_volume_mean`|`float64`|Mean of bar-to-bar deltas|`np.mean(bars['delta_divergence_sell_volume'])`|NaN if insufficient|
|`delta_divergence_imbalance_signed_ratio_mean`|`float64`|Mean of bar-to-bar deltas|`np.mean(bars['delta_divergence_imbalance_signed_ratio'])`|NaN if insufficient|
|`delta_divergence_buy_ratio_mean`|`float64`|Mean of bar-to-bar deltas|`np.mean(bars['delta_divergence_buy_ratio'])`|NaN if insufficient|

---

## 8. Temporal / Tempo Structure

### 8.1 Time Elapsed Metrics

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`time_elapsed_ns_mean`|`float64`|Mean bar duration|`np.mean(bars['time_elapsed_ns'])`|NaN if num_bars=0|
|`time_elapsed_ns_std`|`float64`|Std of bar durations|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`time_elapsed_ns_min`|`uint64`|Min bar duration|`np.min(bars['time_elapsed_ns'])`|0|
|`time_elapsed_ns_max`|`uint64`|Max bar duration|`np.max(bars['time_elapsed_ns'])`|0|

### 8.2 Pace of Contracts Traded

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`pace_of_contracts_traded`|`float64`|Bucket-level: total vol / total time|`volume_total_sum / time_elapsed_ns_total`|NaN if time=0|
|`pace_of_contracts_traded_mean`|`float64`|Mean of bar-level pace|`np.mean(bars['pace_of_contracts_traded'])`|NaN if insufficient|
|`pace_of_contracts_traded_std`|`float64`|Std of bar-level pace|`np.std(..., ddof=1)`|0.0 if num_bars≤1|

### 8.3 Temporal Log Deltas

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`delta_time_elapsed_ns_log_mean`|`float64`|Mean of log deltas|`np.mean(bars['delta_time_elapsed_ns_log'])`|NaN if insufficient|
|`delta_time_elapsed_ns_log_std`|`float64`|Std of log deltas|`np.std(..., ddof=1)`|0.0 if num_bars≤1|
|`delta_pace_of_contracts_traded_log_mean`|`float64`|Mean of log deltas|`np.mean(bars['delta_pace_of_contracts_traded_log'])`|NaN if insufficient|
|`delta_pace_of_contracts_traded_log_std`|`float64`|Std of log deltas|`np.std(..., ddof=1)`|0.0 if num_bars≤1|

### 8.4 Derived Tempo Metrics

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`tempo_stability`|`float64`|Std of time_elapsed_ns_log deltas|Same as `delta_time_elapsed_ns_log_std`|0.0|
|`tempo_acceleration`|`float64`|Slope of pace_log deltas over bar index|Linear regression slope|0.0 if num_bars≤1|

---

## 9. Directional Signals

### 9.1 Active Directional Signals

Based on `active_midpoint_vwap` direction.

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`active_pct_bars_positive_direction`|`float64`|Fraction of bars with positive direction|`count(active_direction_positive) / num_bars`|NaN if num_bars=0|
|`active_directional_streak_max`|`uint32`|Longest streak of same direction|Streak calculation|0|
|`active_directional_reversals_count`|`uint32`|Count of direction changes|`sum(direction_changes)`|0|
|`active_net_direction`|`float64`|(up - down) / num_bars|Direct calc|NaN if num_bars=0|

### 9.2 Adjusted Directional Signals

Based on `adjusted_midpoint_vwap` direction.

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`adjusted_pct_bars_positive_direction`|`float64`|Fraction of bars with positive direction|`count(adjusted_direction_positive) / num_bars`|NaN if num_bars=0|
|`adjusted_directional_streak_max`|`uint32`|Longest streak of same direction|Streak calculation|0|
|`adjusted_directional_reversals_count`|`uint32`|Count of direction changes|`sum(direction_changes)`|0|
|`adjusted_net_direction`|`float64`|(up - down) / num_bars|Direct calc|NaN if num_bars=0|

### 9.3 Passive Directional Signals

Based on `passive_midprice` direction.

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`passive_pct_bars_positive_direction`|`float64`|Fraction of bars with positive direction|`count(passive_direction_positive) / num_bars`|NaN if num_bars=0|
|`passive_directional_streak_max`|`uint32`|Longest streak of same direction|Streak calculation|0|
|`passive_directional_reversals_count`|`uint32`|Count of direction changes|`sum(direction_changes)`|0|
|`passive_net_direction`|`float64`|(up - down) / num_bars|Direct calc|NaN if num_bars=0|

---

## 10. Volatility and Stability Overlays

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`active_price_volatility`|`float64`|Std of active midpoint log deltas|Same as `delta_active_midpoint_vwap_log_std`|0.0|
|`active_spread_volatility`|`float64`|Std of active spread log deltas|Same as `delta_active_spread_vwap_log_std`|0.0|
|`active_return_volatility`|`float64`|Alias for price_volatility|Same as `active_price_volatility`|0.0|
|`active_intermediate_volatility`|`float64`|Alias for price_volatility|Same as `active_price_volatility`|0.0|
|`active_spread_stability`|`float64`|Alias for spread_volatility|Same as `active_spread_volatility`|0.0|
|`active_flow_volatility`|`float64`|Std of volume_total log deltas|Same as `delta_volume_total_log_std`|0.0|
|`adjusted_price_volatility`|`float64`|Std of adjusted midpoint log deltas|Same as `delta_adjusted_midpoint_vwap_log_std`|0.0|
|`adjusted_spread_volatility`|`float64`|Std of adjusted spread log deltas|Same as `delta_adjusted_spread_vwap_log_std`|0.0|
|`passive_price_volatility`|`float64`|Std of passive midprice log deltas|Same as `delta_passive_midprice_log_std`|0.0|

---

## 11. Inter-Bucket Deltas (Regime Transitions)

These are calculated in `calculate_bucket_deltas()` comparing current bucket to previous bucket.

### 11.1 Active Inter-Bucket Deltas

|Field|Type|Description|Transform|Edge Case|
|---|---|---|---|---|
|`bucket_delta_active_spread_vwap_log`|`float64`|Change in active spread|Log ratio|NaN if invalid|
|`bucket_delta_active_spread_vwap_std_log`|`float64`|Change in spread variability|Log ratio|NaN if invalid|
|`bucket_delta_active_spread_volatility`|`float64`|Change in spread volatility|Diff|NaN if invalid|
|`bucket_delta_active_return_volatility_log`|`float64`|Change in return volatility|Log ratio|NaN if invalid|
|`bucket_delta_active_intermediate_volatility_log`|`float64`|Change in intermediate volatility|Log ratio|NaN if invalid|
|`bucket_delta_active_flow_volatility_log`|`float64`|Change in flow volatility|Log ratio|NaN if invalid|
|`bucket_delta_active_pace_of_contracts_traded_log`|`float64`|Change in pace|Log ratio|NaN if invalid|
|`bucket_delta_active_tempo_stability`|`float64`|Change in tempo stability|Diff|NaN if invalid|
|`bucket_delta_active_time_elapsed_ns_total_log`|`float64`|Change in bucket duration|Log ratio|NaN if invalid|
|`bucket_delta_active_imbalance_signed_ratio_atanh`|`float64`|Change in imbalance (transformed)|Atanh diff|NaN if invalid|
|`bucket_delta_active_imbalance_abs_ratio_logit`|`float64`|Change in abs imbalance (transformed)|Logit diff|NaN if invalid|
|`bucket_delta_active_imbalance_persistence`|`float64`|Change in persistence|Diff|NaN if invalid|
|`bucket_delta_active_cumulative_signed_imbalance_norm`|`float64`|Change in normalized cumulative|Diff|NaN if invalid|
|`bucket_delta_active_imbalance_volatility`|`float64`|Change in imbalance volatility|Diff|NaN if invalid|
|`bucket_delta_active_pct_bars_positive_direction`|`float64`|Change in directional bias|Diff|NaN if invalid|
|`bucket_delta_active_directional_streak_max`|`int32`|Change in streak length|Diff|0|
|`bucket_delta_active_directional_reversals_count`|`int32`|Change in reversal count|Diff|0|

### 11.2 Adjusted Inter-Bucket Deltas

|Field|Type|Description|Transform|Edge Case|
|---|---|---|---|---|
|`bucket_delta_adjusted_spread_vwap_log`|`float64`|Change in adjusted spread|Log ratio|NaN if invalid|
|`bucket_delta_adjusted_spread_volatility`|`float64`|Change in spread volatility|Diff|NaN if invalid|
|`bucket_delta_adjusted_return_volatility_log`|`float64`|Change in return volatility|Log ratio|NaN if invalid|
|`bucket_delta_adjusted_imbalance_signed_ratio_atanh`|`float64`|Change in imbalance (transformed)|Atanh diff|NaN if invalid|
|`bucket_delta_adjusted_imbalance_abs_ratio_logit`|`float64`|Change in abs imbalance (transformed)|Logit diff|NaN if invalid|
|`bucket_delta_adjusted_imbalance_persistence`|`float64`|Change in persistence|Diff|NaN if invalid|
|`bucket_delta_adjusted_imbalance_volatility`|`float64`|Change in imbalance volatility|Diff|NaN if invalid|

### 11.3 Second-Order Deltas (Acceleration)

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`bucket_delta2_active_return_volatility_log`|`float64`|Acceleration of volatility change|`delta_current - delta_previous`|NaN if <3 buckets|
|`bucket_delta2_active_imbalance_signed_ratio_atanh`|`float64`|Acceleration of imbalance change|`delta_current - delta_previous`|NaN if <3 buckets|
|`bucket_delta2_active_pace_of_contracts_traded_log`|`float64`|Acceleration of pace change|`delta_current - delta_previous`|NaN if <3 buckets|
|`bucket_delta2_adjusted_return_volatility_log`|`float64`|Acceleration of volatility change|`delta_current - delta_previous`|NaN if <3 buckets|
|`bucket_delta2_adjusted_imbalance_signed_ratio_atanh`|`float64`|Acceleration of imbalance change|`delta_current - delta_previous`|NaN if <3 buckets|

---

## 12. Debug / Diagnostic

|Field|Type|Description|Calculation|Edge Case|
|---|---|---|---|---|
|`bar_ids_checksum`|`uint64`|Sum of bar IDs|`np.sum(bars['id'])`|0|
|`missing_bars_flag`|`bool`|Gap detected in bar sequence|Check continuity|False|
|`processing_time_ns`|`uint64`|Bucket calculation time|Measured|0|
|`nan_count_active_midpoint`|`uint32`|Bars with NaN active midpoint|`np.sum(np.isnan(bars['active_midpoint_vwap']))`|0|
|`nan_count_adjusted_midpoint`|`uint32`|Bars with NaN adjusted midpoint|`np.sum(np.isnan(bars['adjusted_midpoint_vwap']))`|0|
|`nan_count_passive_midprice`|`uint32`|Bars with NaN passive midprice|`np.sum(np.isnan(bars['passive_midprice']))`|0|

---

## 13. Calculation Order

```
Step 1: Meta / Structural
    - id, bucket_type, num_bars, all_bars_complete, bar_volume_size
    - contract_roll_any, contract_roll_count, latest_instrument_id
    - start_ts_ns, end_ts_ns, time_elapsed_ns_total
    - gap_return_any, gap_return_count
    - contains_oversized_order_any, contains_oversized_order_count

Step 2: Order & Volume Statistics (Total/Ambiguous)
    - volume_total_*, order_count_*, order_splits_*
    - delta_volume_total_log_*, delta_order_count_log_*

Step 3: Active Metrics
    - Order counts (buy, sell, none)
    - Volumes (buy, sell, none)
    - Volume/order count log deltas
    - Imbalances and ratios
    - Link function transforms
    - Imbalance deltas
    - Derived imbalance metrics
    - VWAPs (buy, sell, none, spread, midpoint)
    - Weighted midpoints
    - Price log deltas
    - Weighted midpoint log deltas
    - Price range metrics
    - Pace metrics
    - N-side inferred aggregations

Step 4: Adjusted Metrics
    - Volumes
    - Imbalances and ratios
    - Link function transforms
    - Imbalance deltas
    - Derived imbalance metrics
    - VWAPs
    - Weighted midpoints
    - Price log deltas
    - Weighted midpoint log deltas

Step 5: Passive Metrics
    - Midprice
    - Midprice deltas
    - CDF statistics
    - Volume classifications
    - Imbalances and ratios
    - Link function transforms
    - Imbalance deltas
    - Derived imbalance metrics

Step 6: Divergence Metrics
    - Aggregated bar-level divergences
    - Bucket-level divergences
    - Divergence deltas

Step 7: Temporal / Tempo Structure
    - Time elapsed metrics
    - Pace of contracts traded
    - Temporal log deltas
    - Derived tempo metrics

Step 8: Directional Signals
    - Active directional signals
    - Adjusted directional signals
    - Passive directional signals

Step 9: Volatility and Stability Overlays
    - Active volatility metrics
    - Adjusted volatility metrics
    - Passive volatility metrics

Step 10: Debug / Diagnostic
    - Checksums and counts
```

**Note:** Inter-bucket deltas are calculated in `calculate_bucket_deltas()`.

---

## 14. Edge Case Handling Summary

|Scenario|Affected Variables|Fill Value|Rationale|
|---|---|---|---|
|No bars (num_bars=0)|All|Raise ValueError|Cannot calculate statistics for empty bucket|
|Single bar (num_bars=1)|All std fields|0.0|No variance with single observation|
|num_bars≤2|Skewness fields|0.0|Insufficient data for skewness|
|No buy trades|Buy-specific VWAPs, prices, ratios|NaN|Undefined; missing data|
|No sell trades|Sell-specific VWAPs, prices, ratios|NaN|Undefined; missing data|
|No N-side trades|N-side specific metrics|0 for counts/volumes, NaN for prices|Count is known (zero); price undefined|
|Division by zero (ratios)|All ratio metrics|NaN|Mathematically undefined|
|time_elapsed_ns_total=0|Pace metrics|NaN|Division by zero|
|Bucket sign=0|Persistence metrics|0.0|Cannot determine persistence with neutral sign|
|Log of zero/negative|Log-transformed metrics|NaN|Mathematically undefined|
|Previous bucket missing|Inter-bucket deltas|NaN|No comparison possible|
|<3 buckets in history|Second-order deltas|NaN|Insufficient history for acceleration|

---

_End of Schema Reference_